In [0]:
from pyspark.sql.functions import sum, desc, concat, lit , col

results = spark.read.table("f1_processed.results")
drivers = spark.read.table("f1_processed.drivers")
races = spark.read.table("f1_processed.races")
status = spark.read.table("f1_processed.status")
sprint = spark.read.table("f1_processed.sprint_results")


f1_detailed_results = results.join(drivers, "driver_id") \
                             .join(races, "race_id") \
                             .join(status, "status_id") \
                             .select(
                                 col("year"),
                                 col("name").alias("race_name"),
                                 concat(col("forename"), lit(" "), col("surname")).alias("driver_name"),
                                 col("points"),
                                 col("status").alias("finish_status")
                             )

bad_luck_drivers = f1_detailed_results.filter(col("finish_status").isin("Engine", "Gearbox", "Transmission", "Electrical")) \
    .groupBy("driver_name") \
    .count() \
    .orderBy(desc("count"))


total_points_df = results.groupBy("driver_id").agg(sum("points").alias("race_points")) \
    .join(sprint.groupBy("driver_id").agg(sum("points").alias("sprint_points")), "driver_id", "left") \
    .fillna(0) 


f1_detailed_results.write.mode("overwrite").saveAsTable("f1_presentation.detailed_results")
bad_luck_drivers.write.mode("overwrite").saveAsTable("f1_presentation.driver_bad_luck_stats")

print( "Final transformations complete!")
bad_luck_drivers.show(5)


Final transformations complete!
+-----------------+-----+
|      driver_name|count|
+-----------------+-----+
|Andrea de Cesaris|   60|
| Riccardo Patrese|   56|
| Michele Alboreto|   54|
|   Gerhard Berger|   41|
|    Nigel Mansell|   41|
+-----------------+-----+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import sum, desc
detailed_results = spark.read.table('f1_presentation.detailed_results')

top_drivers_of_all_time=detailed_results.groupBy('driver_name').agg(sum('points').alias('total_points')).orderBy(desc('total_points'))

top_drivers_of_all_time.write.mode('overwrite').saveAsTable('f1_presentation.top_drivers_all_time')

In [0]:
top_drivers_of_all_time.show(10)

+------------------+------------+
|       driver_name|total_points|
+------------------+------------+
|    Lewis Hamilton|      4955.5|
|    Max Verstappen|      3301.5|
|  Sebastian Vettel|      3098.0|
|   Fernando Alonso|      2380.0|
|    Kimi Räikkönen|      1873.0|
|   Valtteri Bottas|      1788.0|
|      Nico Rosberg|      1594.5|
|   Charles Leclerc|      1588.0|
|      Sergio Pérez|      1585.0|
|Michael Schumacher|      1566.0|
+------------------+------------+
only showing top 10 rows
